# 09 - MRT Parsing and Inspection

Controlled pilot step for the 9 TRAIN incidents only. This notebook documents the parser, raw BGP UPDATE observation counts, representative records, incident relevance checks, timestamp coverage, and parser/data quality audit.

This notebook does not build 5-minute feature windows, create Normal labels, add CAIDA/RPKI features, or train models.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

parser_env = pd.read_csv(PILOT / 'mrt_parser_environment_v1.csv')
counts = pd.read_csv(PILOT / 'raw_bgp_message_counts_v1.csv')
sample = pd.read_csv(PILOT / 'parsed_bgp_sample_v1.csv')
relevance = pd.read_csv(PILOT / 'incident_relevance_summary_v1.csv')
coverage = pd.read_csv(PILOT / 'mrt_parsed_time_coverage_v1.csv')
audit = pd.read_csv(PILOT / 'mrt_parsing_audit_v1.csv')

## Parser Environment

In [ ]:
parser_env

## Raw Message and Prefix-Level Observation Counts

`bgpdump -q -m` emits one pipe-delimited row per prefix-level announcement or withdrawal observation. These are raw parsed observations, not feature rows.

In [ ]:
counts[counts['aggregation_level'].isin(['incident', 'all_pilot'])]

## Representative Parsed Records

In [ ]:
sample.groupby([sample['incident_uid'].str.split(':').str[0], 'announcement_or_withdrawal']).size()

In [ ]:
sample.head(10)

## Incident Relevance Inspection

Matching observations are counted using the pilot metadata available for each class. Nonmatching observations are preserved because RouteViews files contain the full collector feed for the acquisition interval.

In [ ]:
relevance

## Timestamp Coverage

In [ ]:
coverage

## Parser/Data Quality Audit

In [ ]:
audit

In [ ]:
status_counts = audit['status'].value_counts().to_dict()
total = counts[counts['incident_uid'] == 'ALL_PILOT_INCIDENTS'].iloc[0]
observable = (relevance['directly_matching_observations'] > 0).all()

print('PASS/REVIEW/FAIL:', status_counts)
print('MRT files:', int(total['mrt_files']))
print('Parsed prefix-level observations:', int(total['total_prefix_level_observations']))
print('All 9 incidents have direct metadata matches:', bool(observable))